# Diabetes Prediction System Using Machine Learning
### End-to-End Clinical Diagnostic Pipeline & Model Benchmarking

**Abstract:**
> *Diabetes is a chronic disease that affects millions of people worldwide and can lead to serious health complications if it is not detected and managed at an early stage. Early prediction of diabetes can help individuals take appropriate preventive measures and receive timely medical care. This project presents a Diabetes Prediction System using Machine Learning to predict whether a person is likely to have diabetes based on relevant medical and demographic attributes.*
>
> *The system uses patient-related features such as Pregnancies, Glucose Level, Blood Pressure, Skin Thickness, Insulin, Body Mass Index (BMI), Diabetes Pedigree Function, and Age. The dataset is preprocessed to handle missing or invalid values, followed by feature analysis and data normalization where required. Machine learning classification algorithms such as Logistic Regression, Decision Tree, Random Forest, Support Vector Machine (SVM), and K-Nearest Neighbors (KNN) can be trained and evaluated to identify the most suitable prediction model.*
>
> *The performance of the models is assessed using evaluation metrics including accuracy, precision, recall, F1-score, and confusion matrix. The proposed system can provide a simple and efficient way to identify individuals who may be at risk of diabetes. It is intended as a predictive decision-support tool and not as a replacement for professional medical diagnosis. The project demonstrates how machine learning can be applied to healthcare data to support early risk prediction and improve preventive healthcare.*

## 1. Import Essential Libraries
Importing scientific computing, visualization, and machine learning modules from Scikit-Learn.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, confusion_matrix,
    classification_report
)

# Visual styling
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
print("Libraries imported successfully!")

## 2. Load Dataset & Clinical Attributes Exploration
The dataset used is the Pima Indians Diabetes dataset (768 patient records with 8 diagnostic features).

In [ ]:
data_path = '../data/raw/diabetes.csv' if os.path.exists('../data/raw/diabetes.csv') else 'diabetes.csv'
df = pd.read_csv(data_path)

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
display(df.head())
print("\nData Types & Non-Null Counts:")
df.info()

In [ ]:
print("Descriptive Statistical Summary:")
display(df.describe().T)

# Class balance
class_counts = df['Outcome'].value_counts()
print(f"\nClass Distribution:\n0 (Non-Diabetic): {class_counts[0]} ({class_counts[0]/len(df)*100:.1f}%)\n1 (Diabetic):     {class_counts[1]} ({class_counts[1]/len(df)*100:.1f}%)")

## 3. Exploratory Data Analysis (EDA)
Visualizing the class distribution, feature correlations, and biomarker comparisons between diabetic and non-diabetic groups.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class Distribution Pie Chart
axes[0].pie(class_counts, labels=['Non-Diabetic (0)', 'Diabetic (1)'], autopct='%1.1f%%',
            colors=['#3B82F6', '#EF4444'], startangle=140, explode=[0, 0.08])
axes[0].set_title('Target Outcome Proportion', fontsize=13, fontweight='bold')

# Glucose vs Outcome
sns.boxplot(x='Outcome', y='Glucose', data=df, palette=['#3B82F6', '#EF4444'], ax=axes[1])
axes[1].set_title('Plasma Glucose Level by Diagnostic Outcome', fontsize=13, fontweight='bold')
axes[1].set_xticklabels(['Non-Diabetic (0)', 'Diabetic (1)'])

plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Clinical Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Rigorous Data Preprocessing

### 4.1 Identification of Physiological Zero Values
In a living human subject, `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI` **cannot biologically be zero**.
These entries indicate unmeasured or missing laboratory tests and are converted to `np.nan`.

In [ ]:
zero_invalid_features = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
print("Count of invalid zero values per feature:")
for col in zero_invalid_features:
    num_zeros = (df[col] == 0).sum()
    print(f"  • {col:15s}: {num_zeros:3d} zeros ({num_zeros / len(df) * 100:.1f}% missing)")

# Replace biological zeros with NaN
df_preprocessed = df.copy()
for col in zero_invalid_features:
    df_preprocessed[col] = df_preprocessed[col].replace(0, np.nan)

### 4.2 Stratified Train-Test Split (80% Train / 20% Test)
We split the data **before** calculating imputation and scaling parameters to strictly prevent **data leakage**.

In [ ]:
X = df_preprocessed.drop('Outcome', axis=1)
y = df_preprocessed['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape} (Diabetic ratio: {y_train.mean():.2%})")
print(f"Testing set shape:  {X_test.shape} (Diabetic ratio: {y_test.mean():.2%})")

### 4.3 Median Imputation, IQR Outlier Capping & Standardization ($Z$-Score)
1. **Median Imputation**: Imputes `NaN` using median values learned solely on the training partition.
2. **IQR Boundary Capping**: Clips extreme outliers within $[Q_1 - 1.5 \times \text{IQR}, Q_3 + 1.5 \times \text{IQR}]$.
3. **StandardScaler**: Normalizes feature dimensions to zero mean and unit variance $(\mu=0, \sigma=1)$.

In [ ]:
feature_names = X.columns.tolist()

# 1. Median Imputation
imputer = SimpleImputer(strategy='median')
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=feature_names, index=X_train.index)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=feature_names, index=X_test.index)

# 2. IQR Outlier Bounds Calculation on Train Set
iqr_bounds = {}
for col in feature_names:
    q1 = X_train_imp[col].quantile(0.25)
    q3 = X_train_imp[col].quantile(0.75)
    iqr = q3 - q1
    lb = q1 - 1.5 * iqr
    ub = q3 + 1.5 * iqr
    iqr_bounds[col] = (lb, ub)
    X_train_imp[col] = X_train_imp[col].clip(lower=lb, upper=ub)
    X_test_imp[col] = X_test_imp[col].clip(lower=lb, upper=ub)

# 3. Z-score Standardization
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_imp), columns=feature_names, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_imp), columns=feature_names, index=X_test.index)

print("Preprocessing pipeline successfully applied without data leakage!")
display(X_train_scaled.head())

## 5. Machine Learning Classification Models Training
We instantiate and fit all 5 core classification algorithms designated in the project abstract:
1. **Logistic Regression (LR)**
2. **Decision Tree Classifier (DT)**
3. **Random Forest Classifier (RF)**
4. **Support Vector Machine (SVM with RBF kernel)**
5. **K-Nearest Neighbors (KNN)**

Plus an advanced **Weighted Soft Voting Ensemble** combining all base models.

In [ ]:
base_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, min_samples_split=4, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=6, random_state=42),
    'Support Vector Machine': SVC(kernel='rbf', probability=True, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=7, weights='distance')
}

# 5-Fold Stratified Cross Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

print("Performing 5-Fold Stratified Cross Validation on Training Data:")
for name, model in base_models.items():
    acc_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
    roc_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='roc_auc')
    cv_results[name] = {
        'cv_acc_mean': acc_scores.mean(),
        'cv_acc_std': acc_scores.std(),
        'cv_auc_mean': roc_scores.mean(),
        'cv_auc_std': roc_scores.std()
    }
    model.fit(X_train_scaled, y_train)
    print(f"  • {name:25s} | CV Accuracy: {acc_scores.mean()*100:.2f}% | CV ROC-AUC: {roc_scores.mean():.4f}")

# Build Weighted Ensemble based on CV ROC-AUC weights
ensemble_estimators = [(name, model) for name, model in base_models.items()]
weights = [cv_results[name]['cv_auc_mean'] for name in base_models.keys()]

ensemble_model = VotingClassifier(estimators=ensemble_estimators, voting='soft', weights=weights)
ensemble_model.fit(X_train_scaled, y_train)

ens_acc = cross_val_score(ensemble_model, X_train_scaled, y_train, cv=cv, scoring='accuracy').mean()
ens_auc = cross_val_score(ensemble_model, X_train_scaled, y_train, cv=cv, scoring='roc_auc').mean()
cv_results['Weighted Ensemble (Proposed)'] = {'cv_acc_mean': ens_acc, 'cv_acc_std': 0.0, 'cv_auc_mean': ens_auc, 'cv_auc_std': 0.0}

all_models = {**base_models, 'Weighted Ensemble (Proposed)': ensemble_model}
print(f"  • {'Weighted Ensemble (Proposed)':25s} | CV Accuracy: {ens_acc*100:.2f}% | CV ROC-AUC: {ens_auc:.4f}")

## 6. Model Evaluation on Unseen Test Partition
Benchmarking all models on the held-out test dataset ($N=154$) across:
- **Accuracy**
- **Precision**
- **Recall (Sensitivity)**
- **Specificity**
- **F1-Score**
- **ROC-AUC Score**

In [ ]:
benchmark_list = []

for name, model in all_models.items():
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp)
    
    benchmark_list.append({
        'Model': name,
        'CV Accuracy': f"{cv_results[name]['cv_acc_mean']*100:.2f}%",
        'CV ROC-AUC': f"{cv_results[name]['cv_auc_mean']:.4f}",
        'Test Accuracy': f"{accuracy_score(y_test, y_pred)*100:.2f}%",
        'Precision': f"{precision_score(y_test, y_pred)*100:.2f}%",
        'Recall (Sensitivity)': f"{recall_score(y_test, y_pred)*100:.2f}%",
        'Specificity': f"{specificity*100:.2f}%",
        'F1-Score': round(f1_score(y_test, y_pred), 4),
        'ROC-AUC': round(roc_auc_score(y_test, y_prob), 4)
    })

results_df = pd.DataFrame(benchmark_list).sort_values(by='ROC-AUC', ascending=False)
print("\n--- COMPREHENSIVE PERFORMANCE BENCHMARK ---")
display(results_df)


## 7. Performance Visualizations: Confusion Matrices & ROC Curves

In [ ]:
# ROC Curves Comparison
plt.figure(figsize=(10, 7))
for name, model in all_models.items():
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_val = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc_val:.3f})", linewidth=2.0)

plt.plot([0, 1], [0, 1], 'k--', label='Chance Level (AUC = 0.50)')
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Recall / Sensitivity)', fontsize=12)
plt.title('Multi-Model ROC Curves Comparison', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', frameon=True)
plt.show()

In [ ]:
# Confusion Matrices Subplot Grid
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.ravel()

for idx, (name, model) in enumerate(all_models.items()):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False,
                xticklabels=['Non-Diabetic', 'Diabetic'],
                yticklabels=['Non-Diabetic', 'Diabetic'])
    axes[idx].set_title(name, fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Predicted Label')
    axes[idx].set_ylabel('Actual Label')

plt.tight_layout()
plt.show()

In [ ]:
# Random Forest Feature Importance
rf_model = all_models['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=feature_names).sort_values(ascending=True)

plt.figure(figsize=(9, 5))
importances.plot(kind='barh', color='#2563EB')
plt.title('Clinical Feature Importance (Random Forest Classifier)', fontsize=13, fontweight='bold')
plt.xlabel('Gini Relative Importance Score')
plt.tight_layout()
plt.show()

## 8. Clinical Decision Support: Live Patient Prediction Function
Demonstrating inference on individual patient diagnostic profiles with risk stratification.

In [ ]:
def assess_patient(patient_dict, model_choice='Weighted Ensemble (Proposed)'):
    """
    Predicts diabetes risk for an individual patient dictionary.
    """
    df_pt = pd.DataFrame([patient_dict])[feature_names]
    for col in zero_invalid_features:
        if col in df_pt.columns:
            df_pt[col] = df_pt[col].replace(0, np.nan)
    
    # Impute
    df_pt_imp = pd.DataFrame(imputer.transform(df_pt), columns=feature_names)
    # Clip outliers
    for col in feature_names:
        lb, ub = iqr_bounds[col]
        df_pt_imp[col] = df_pt_imp[col].clip(lower=lb, upper=ub)
    # Scale
    df_pt_scaled = pd.DataFrame(scaler.transform(df_pt_imp), columns=feature_names)
    
    chosen_model = all_models[model_choice]
    prediction = chosen_model.predict(df_pt_scaled)[0]
    proba = chosen_model.predict_proba(df_pt_scaled)[0, 1]
    
    if proba < 0.35:
        risk_tier = "Low Risk (Routine monitoring)"
    elif proba < 0.65:
        risk_tier = "Moderate Risk (Dietary intervention & re-testing recommended)"
    else:
        risk_tier = "High Risk (Urgent clinical laboratory follow-up recommended)"
        
    return {
        'Diagnostic Prediction': 'DIABETIC' if prediction == 1 else 'NON-DIABETIC',
        'Risk Probability': f"{proba * 100:.1f}%",
        'Clinical Risk Tier': risk_tier,
        'Model Applied': model_choice
    }

# Sample Test Cases
case_low = {'Pregnancies': 1, 'Glucose': 88.0, 'BloodPressure': 66.0, 'SkinThickness': 23.0, 'Insulin': 94.0, 'BMI': 22.5, 'DiabetesPedigreeFunction': 0.23, 'Age': 24.0}
case_high = {'Pregnancies': 6, 'Glucose': 175.0, 'BloodPressure': 84.0, 'SkinThickness': 35.0, 'Insulin': 210.0, 'BMI': 36.8, 'DiabetesPedigreeFunction': 0.85, 'Age': 52.0}

print("=== ASSESSMENT RESULT: CASE 1 (Healthy Profile) ===")
for k, v in assess_patient(case_low).items():
    print(f"  {k:25s}: {v}")

print("\n=== ASSESSMENT RESULT: CASE 2 (High-Risk Profile) ===")
for k, v in assess_patient(case_high).items():
    print(f"  {k:25s}: {v}")